# 04 Well tubing: wellhead pressure for hydrogen vs methane at Rough

Adds the tubing (bottomhole ↔ wellhead, `h2_bottleneck.tubing`) to the inflow model of notebook 03, and gives the wellhead pressure the surface must supply (injection) or can receive (withdrawal). Method: the average temperature and Z method as published by Scafidi et al. (2023, eq. 12), in SI units (see the methodology). Validation (V2) is in notebook 04b.

**Outputs**
1. Gas column (printed only)
2. Headline: wellhead pressure needed for the required field rates → `04_whp_at_required_rate.csv` + figure
3. Subsurface curve at node 3 → `04_node3_subsurface_curves.csv` + figure (input to the network model)
4. Tubing-size sensitivity → `04_tubing_size_sensitivity.csv`

In [ ]:
%load_ext autoreload
%autoreload 2
import subprocess, hashlib, datetime, pathlib
root = pathlib.Path.cwd().parent
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=root).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=root).stdout.strip() != ""
inp = hashlib.sha256((root / "data" / "inputs.csv").read_bytes()).hexdigest()[:10]
wls = hashlib.sha256((root / "data" / "wells.csv").read_bytes()).hexdigest()[:10]
print(f"Run {datetime.datetime.now():%Y-%m-%d %H:%M} | commit {commit}{' (UNCOMMITTED CHANGES)' if dirty else ''} | inputs.csv {inp} | wells.csv {wls}")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from h2_bottleneck import inputs, props
from h2_bottleneck import well as wl
from h2_bottleneck import tubing as tb

FIG, TAB = root / "results" / "figures", root / "results" / "tables"
GASES = ("Hydrogen", "Methane")
COL = {"Hydrogen": "tab:blue", "Methane": "tab:orange"}
DAY, MSM3, BAR, IN = 86400.0, 1e6, 1e5, 0.0254

x = inputs.load()
f = wl.Formation.from_inputs(x)
wells = [wl.Well.from_row(r) for _, r in inputs.load_wells().iterrows()]
tub = tb.Tubing.from_inputs(x, f)
tubs = {w.name: tub for w in wells}                 # A15: same tubing in every well
m = {g: wl.Pseudopressure(f.T, g) for g in GASES}
print(f"Tubing: {tub.tvd:.0f} m vertical, ID {tub.d / IN:.1f} in, T_wh {tub.T_wh - 273.15:.0f} °C, T_res {tub.T_bh - 273.15:.1f} °C")

## 1. Gas column with no flow

p_wf − p_wh at q = 0 is the weight of the gas column. Hydrogen's is far lighter, so for injection the wellhead must supply nearly the full bottomhole pressure.

In [ ]:
for p_wh in (150e5, 250e5):
    col = {g: (tb.pwf_from_pwh(p_wh, 0.0, tub, g) - p_wh) / BAR for g in GASES}
    print(f"WHP {p_wh / BAR:.0f} bar: column H2 {col['Hydrogen']:.1f} bar, CH4 {col['Methane']:.1f} bar "
          f"(ratio {col['Methane'] / col['Hydrogen']:.1f})")

## 2. Wellhead pressure needed for the required field rates (headline)

The common wellhead (manifold) pressure at which the 29 wells together move the field rate the storage cycle needs (`tubing.manifold_whp_for_rate`), each well held at its BHP limit if it would pass it:
- **H2:** notebook 02 averages, 21.1 million Sm³/d injection and 28.1 million Sm³/d withdrawal;
- **CH4 same volume:** the same Sm³/d (isolates the gas column and friction);
- **CH4 same energy:** the Sm³/d carrying the same HHV energy.

NaN = not reachable with the wells at their BHP limits.

In [ ]:
Q_H2 = {"injection": -21.1, "withdrawal": 28.1}                    # million Sm3/d, notebook 02
e_ratio = (props.rho_sc("Hydrogen") * x["hhv_hydrogen"]) / (props.rho_sc("Methane") * x["hhv_methane"])
cases = [("Hydrogen", "H2", 1.0), ("Methane", "CH4 same volume", 1.0), ("Methane", "CH4 same energy", e_ratio)]
rows = []
t0 = time.time()
for pr in np.arange(180e5, 271e5, 15e5):
    for mode, q_h2 in Q_H2.items():
        for g, label, scale in cases:
            q = q_h2 * scale * MSM3 / DAY
            rows.append(dict(p_res_bar=pr / BAR, mode=mode, case=label, q_MSm3d=q * DAY / MSM3,
                             p_wh_bar=tb.manifold_whp_for_rate(q, pr, wells, f, tubs, g, m[g])["p_wh"] / BAR))
print(f"Energy per Sm3, H2/CH4 = {e_ratio:.3f};  solved in {time.time() - t0:.0f} s")
req = pd.DataFrame(rows)
req.to_csv(TAB / "04_whp_at_required_rate.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sty = {"H2": ("tab:blue", "-"), "CH4 same volume": ("tab:orange", "-"), "CH4 same energy": ("tab:orange", "--")}
for a, mode in zip(ax, Q_H2):
    for label, (c, ls) in sty.items():
        d = req[(req["mode"] == mode) & (req.case == label)]
        a.plot(d.p_res_bar, d.p_wh_bar, color=c, ls=ls, marker="o", label=label)
    a.set_title(f"{mode.capitalize()}: {abs(Q_H2[mode])} million Sm$^3$/d H$_2$")
    a.set_xlabel("Average reservoir pressure (bar)"); a.set_ylabel("Wellhead pressure (bar)"); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "04_whp_at_required_rate.png", dpi=300)
req.pivot_table(index=["mode", "p_res_bar"], columns="case", values="p_wh_bar").round(1)

## 3. Subsurface curve at node 3

For each common wellhead pressure, the field rate from all 29 wells (inflow + tubing + BHP limits). Withdrawal > 0, injection < 0. The surface model (pipeline and compression) will intersect this curve.

In [ ]:
rows = []
t0 = time.time()
for pr in (180e5, 220e5, 270e5):
    for g in GASES:
        for p3 in np.arange(50e5, 331e5, 20e5):
            r = tb.manifold_rate(p3, pr, wells, f, tubs, g, m[g])
            rows.append(dict(p_res_bar=pr / BAR, gas=g, p_wh_bar=p3 / BAR, q_field_MSm3d=r["q"] * DAY / MSM3,
                             n_bhp_limited=r["n_bhp_limited"]))
print(f"Node-3 curves in {time.time() - t0:.0f} s")
node3 = pd.DataFrame(rows)
node3.to_csv(TAB / "04_node3_subsurface_curves.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
for pr, ls in zip((180.0, 220.0, 270.0), (":", "-", "--")):
    for g in GASES:
        c = node3[(node3.p_res_bar == pr) & (node3.gas == g)]
        ax.plot(c.q_field_MSm3d, c.p_wh_bar, color=COL[g], ls=ls, label=f"{g}, p$_{{res}}$ {pr:.0f} bar")
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Field rate (million Sm$^3$/d)  ← injection | withdrawal →"); ax.set_ylabel("Wellhead pressure (bar)")
ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(FIG / "04_node3_subsurface_curves.png", dpi=300)

## 4. Tubing-size sensitivity

Rough's tubing size is a data gap (D4). Highest and median WHP needed to hold the wells at BHP_max at p_res = 220 bar, for the 2–8 in sizes of Scafidi et al. (2023). `n_tubing_limited` = wells whose tubing cannot carry their limit rate.

In [ ]:
from dataclasses import replace
rows = []
for d_in in (2.0, 4.0, 6.0, 8.0):
    t_d = replace(tub, d=d_in * IN)
    for g in GASES:
        p = np.array([r["p_wh"] for r in (tb.whp_at_bhp_limit(220e5, w, f, t_d, g, m[g], "injection") for w in wells)
                      if r["q"] != 0.0])
        ok = p[np.isfinite(p)]
        rows.append(dict(tubing_in=d_in, gas=g, p_wh_max_bar=ok.max() / BAR if ok.size else np.nan,
                         p_wh_median_bar=np.median(ok) / BAR if ok.size else np.nan, n_tubing_limited=int(np.isnan(p).sum())))
sens = pd.DataFrame(rows)
sens.to_csv(TAB / "04_tubing_size_sensitivity.csv", index=False)
sens.pivot(index="tubing_in", columns="gas", values=["p_wh_max_bar", "p_wh_median_bar", "n_tubing_limited"]).round(1)

## Findings

_Fill in from the outputs (exact values):_
1. **Gas column** at 150 bar WHP: H2 … bar, CH4 … bar (ratio …).
2. **Headline:** injecting 21.1 million Sm³/d of H2 needs … to … bar at the wellhead across 180–270 bar, against … to … bar for the same volume of CH4 and … to … bar for the same energy.
3. **Withdrawal:** H2 leaves … bar more at the wellhead.
4. **Tubing size:** holding the wells at BHP_max at 220 bar needs … bar (8 in) to … bar (4 in); 2 in is tubing-limited.

**Next:** `pipeline.py`, then the surface curve at node 3 to intersect with section 3.